# W&B Sweeps — Fine-tune the Registered Model

Sweeps are W&B's hyperparameter search. A **sweep controller** samples
hyperparameters from a config, hands them to an **agent** that runs
one training trial per proposal, and updates its beliefs from the
metric each trial logs back. The Bayesian method used here proposes
new points based on which regions of the space look most promising —
it doesn't sweep exhaustively, so it converges much faster than grid
search on modest trial counts.

This notebook demonstrates the full loop:

1. **Pull the champion checkpoint** from the Registry (produced by
   `experiment_tracking.ipynb`) — this is the warm-start.
2. **Fine-tune** it under different hyperparameter proposals via a
   Bayesian sweep.
3. **Promote the sweep winner** back to the Registry under a new
   Collection (`model_hyperparameter_tuned`) so downstream consumers
   can pick it up.

**Prerequisites** — both must be run first:
- `registry.ipynb` — produces `wandb-registry-airfoil_surrogates/datasets:production`.
- `experiment_tracking.ipynb` — produces
  `wandb-registry-airfoil_surrogates/checkpoints:production`.


## 1. Setup

Third-party imports and the section-local constants — no
`physicsx_demo` imports, notebook stands alone.


In [1]:
from __future__ import annotations

import json
import math
import tempfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

device = torch.device("cpu")

ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
BASE_COLLECTION = "checkpoints"
TUNED_COLLECTION = "model_hyperparameter_tuned"

DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
CHECKPOINT_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{BASE_COLLECTION}:production"

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"device = {device}")


torch  = 2.12.1
wandb  = 0.28.0
device = cpu


## 2. Model — same TinyTransolver

The sweep fine-tunes the same architecture the training notebook
produced, so the class definition is inlined verbatim. A `state_dict`
alone cannot reconstitute an `nn.Module`, so this cell is required.

Physics-Attention reference: `docs/explainers/pxtransolver.md`.


In [2]:
class PhysicsAttention(nn.Module):
    """One Physics-Attention block: slice -> attend -> deslice, plus an FFN."""

    def __init__(self, hidden: int, n_slices: int, n_heads: int, ffn_expansion: int = 2):
        super().__init__()
        assert hidden % n_heads == 0, "hidden must be divisible by n_heads"
        self.n_slices = n_slices
        self.n_heads = n_heads
        self.head_dim = hidden // n_heads

        self.slice_proj = nn.Linear(hidden, n_slices)
        self.qkv = nn.Linear(hidden, 3 * hidden)
        self.attn_out = nn.Linear(hidden, hidden)
        self.ffn = nn.Sequential(
            nn.Linear(hidden, ffn_expansion * hidden),
            nn.GELU(),
            nn.Linear(ffn_expansion * hidden, hidden),
        )
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

    def _slice_attend_deslice(self, x: torch.Tensor):
        B, N, C = x.shape
        M, H, D = self.n_slices, self.n_heads, self.head_dim

        w = torch.softmax(self.slice_proj(x), dim=-1)              # (B, N, M)
        numer = torch.einsum("bnm,bnc->bmc", w, x)                  # (B, M, C)
        denom = w.sum(dim=1).unsqueeze(-1) + 1e-8                   # (B, M, 1)
        z = numer / denom                                            # (B, M, C)

        qkv = self.qkv(z).view(B, M, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
        attn = torch.softmax(scores, dim=-1)
        z_out = (attn @ v).transpose(1, 2).contiguous().view(B, M, C)
        z_out = self.attn_out(z_out)

        x_out = torch.einsum("bnm,bmc->bnc", w, z_out)              # (B, N, C)
        return x_out, w

    def forward(self, x: torch.Tensor, return_slice_weights: bool = False):
        attended, w = self._slice_attend_deslice(self.norm1(x))
        x = x + attended
        x = x + self.ffn(self.norm2(x))
        if return_slice_weights:
            return x, w
        return x


class TinyTransolver(nn.Module):
    def __init__(
        self,
        in_dim: int = 5,
        out_dim: int = 3,
        hidden: int = 128,
        n_layers: int = 6,
        n_slices: int = 32,
        n_heads: int = 8,
        ffn_expansion: int = 2,
    ):
        super().__init__()
        self.embed = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([
            PhysicsAttention(hidden, n_slices, n_heads, ffn_expansion)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, features: torch.Tensor, return_slice_weights: bool = False):
        x = self.embed(features)
        all_w = []
        for block in self.blocks:
            if return_slice_weights:
                x, w = block(x, return_slice_weights=True)
                all_w.append(w)
            else:
                x = block(x)
        x = self.out_norm(x)
        out = self.head(x)
        if return_slice_weights:
            return out, all_w
        return out


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 3. Pull the pretrained weights + dataset from W&B

A short `sweep-setup` run consumes both the champion checkpoint and
the synthetic dataset. Using `run.use_artifact(...)` records both as
inputs of this run, so the W&B lineage graph shows the setup step
sourcing from the Registry entry produced by `experiment_tracking.ipynb`.

The checkpoint dict written by that notebook contains
`{state_dict, config, epoch, val_pressure_rmse}` — enough to
reinstantiate `TinyTransolver` and load the weights. We compute
train-only normalisation stats up front so every sweep trial shares
identical inputs.


In [3]:
# ---- Download both artifacts inside a short setup run ----
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-setup",
    name="sweep-setup",
    tags=["notebook", "sweep", "setup"],
) as setup_run:
    
    ckpt_art = setup_run.use_artifact(CHECKPOINT_REGISTRY_PATH)
    ckpt_dir = Path(ckpt_art.download())
    ds_art = setup_run.use_artifact(DATASET_REGISTRY_PATH)
    ds_dir = Path(ds_art.download())

    # ---- Load checkpoint + rebuild model config ----
    ckpt_file = next(ckpt_dir.glob("*.pt"))
    ckpt = torch.load(ckpt_file, map_location="cpu")

    MODEL_KEYS = ("in_dim", "out_dim", "hidden", "n_layers", "n_slices", "n_heads", "ffn_expansion")
    PRETRAINED_CONFIG = {k: ckpt["config"][k] for k in MODEL_KEYS}
    PRETRAINED_STATE_DICT = ckpt["state_dict"]
    
    print(f"loaded champion checkpoint (pretrain val/pressure_rmse = {ckpt.get('val_pressure_rmse'):.4f})")
    print(f"model config: {PRETRAINED_CONFIG}")

    # ---- Load JSON dataset -> feature/target tensors ----
    ds_file = next(ds_dir.glob("*.json"))
    with open(ds_file) as f:
        payload = json.load(f)

    features_list, targets_list = [], []
    for c in payload["cases"]:
        points = np.array(c["points"], dtype=np.float32)                     # (N, 2)
        sdf = np.array(c["sdf"], dtype=np.float32)                           # (N,)
        pressure = np.array(c["pressure"], dtype=np.float32)                 # (N,)
        velocity = np.array(c["velocity"], dtype=np.float32)                 # (N, 2)
        n = points.shape[0]
        aoa_col = np.full((n, 1), c["aoa"], dtype=np.float32)
        log_re_col = np.full((n, 1), np.log10(c["re"]) - 6.0, dtype=np.float32)
        features_list.append(np.concatenate([points, sdf[:, None], aoa_col, log_re_col], axis=1))
        targets_list.append(np.concatenate([pressure[:, None], velocity], axis=1))

    X_all = torch.from_numpy(np.stack(features_list)).float()                # (n_cases, N, 5)
    Y_all = torch.from_numpy(np.stack(targets_list)).float()                 # (n_cases, N, 3)

    # 80/20 split, matching experiment_tracking convention.
    n_cases = X_all.size(0)
    n_val = int(n_cases * 0.2)
    X_train, X_val = X_all[:-n_val], X_all[-n_val:]
    Y_train, Y_val = Y_all[:-n_val], Y_all[-n_val:]

    # Train-only normalisation stats (features and targets).
    FEAT_MEAN = X_train.mean(dim=(0, 1), keepdim=True)
    FEAT_STD = X_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)
    TGT_MEAN = Y_train.mean(dim=(0, 1), keepdim=True)
    TGT_STD = Y_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)

    X_TRAIN_N = ((X_train - FEAT_MEAN) / FEAT_STD).to(device)
    Y_TRAIN_N = ((Y_train - TGT_MEAN) / TGT_STD).to(device)
    X_VAL_N = ((X_val - FEAT_MEAN) / FEAT_STD).to(device)
    Y_VAL_N = ((Y_val - TGT_MEAN) / TGT_STD).to(device)

    print(f"train: X={tuple(X_TRAIN_N.shape)} Y={tuple(Y_TRAIN_N.shape)}    "
        f"val: X={tuple(X_VAL_N.shape)} Y={tuple(Y_VAL_N.shape)}")

    # Baseline sanity check: what val/pressure_rmse does the warm-start hit before any tuning?
    _baseline = TinyTransolver(**PRETRAINED_CONFIG).to(device)
    _baseline.load_state_dict(PRETRAINED_STATE_DICT)
    _baseline.eval()
    with torch.no_grad():
        _err = _baseline(X_VAL_N) - Y_VAL_N
        baseline_p_rmse = _err[..., 0].pow(2).mean().sqrt().item()
    
    print(f"warm-start baseline val/pressure_rmse = {baseline_p_rmse:.4f}   (sweep goal: beat this)")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: jbutt (wandb_emea) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


loaded champion checkpoint (pretrain val/pressure_rmse = 0.6750)
model config: {'in_dim': 5, 'out_dim': 3, 'hidden': 128, 'n_layers': 6, 'n_slices': 32, 'n_heads': 8, 'ffn_expansion': 2}
train: X=(64, 512, 5) Y=(64, 512, 3)    val: X=(16, 512, 5) Y=(16, 512, 3)
warm-start baseline val/pressure_rmse = 0.6750   (sweep goal: beat this)


## 4. Sweep configuration and training function

Four hyperparameters, all training-time — architecture is fixed by
the loaded `state_dict`:

- **`lr`** — dominant fine-tune knob. Log-uniform `[1e-5, 3e-3]`.
- **`weight_decay`** — regularisation on top of already-trained
  weights. Log-uniform `[1e-6, 1e-3]`.
- **`lambda_aux`** — weight on the slice load-balancing aux term.
  Uniform `[0.0, 0.05]`. The original run used `0.01`; the sweep
  tests whether that's still the right weight for fine-tuning.
- **`batch_size`** — discrete `[4, 8, 16]`. Matters for gradient
  noise on the small 64-case train split.

The `train_fn` closure captures the pretrained state and normalised
tensors from §3, so each trial starts from an identical warm-start
without re-downloading.


In [4]:
FINE_TUNE_EPOCHS = 5

SWEEP_CONFIG = {
    "method": "bayes",
    "metric": {"name": "val/pressure_rmse", "goal": "minimize"},
    "parameters": {
        "lr":           {"distribution": "log_uniform_values", "min": 1e-5, "max": 3e-3},
        "weight_decay": {"distribution": "log_uniform_values", "min": 1e-6, "max": 1e-3},
        "lambda_aux":   {"distribution": "uniform",             "min": 0.0,  "max": 0.05},
        "batch_size":   {"values": [4, 8, 16]},
    },
}

def train_fn():
    with wandb.init(
        job_type="sweep-trial",
        tags=["notebook", "sweep", "fine-tune"],
    ) as run:
        cfg = run.config

        # Record source artifacts on this trial too so the lineage graph
        # includes every trial as a consumer of the champion + dataset.
        run.use_artifact(CHECKPOINT_REGISTRY_PATH)
        run.use_artifact(DATASET_REGISTRY_PATH)

        model = TinyTransolver(**PRETRAINED_CONFIG).to(device)
        model.load_state_dict(PRETRAINED_STATE_DICT)
        optim = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.lr,
            weight_decay=cfg.weight_decay,
        )

        n = X_TRAIN_N.size(0)
        val_p_rmse = float("nan")
        for epoch in range(FINE_TUNE_EPOCHS):
            # --- train ---
            model.train()
            perm = torch.randperm(n)
            for start in range(0, n, cfg.batch_size):
                idx = perm[start:start + cfg.batch_size]
                xb, yb = X_TRAIN_N[idx], Y_TRAIN_N[idx]
                pred, all_w = model(xb, return_slice_weights=True)
                mse = F.mse_loss(pred, yb)

                # Two-term aux loss identical to experiment_tracking.
                aux_terms = []
                for w in all_w:
                    p_slice = w.mean(dim=(0, 1)).clamp(min=1e-8)
                    balance = -(-(p_slice * p_slice.log()).sum())
                    w_c = w.clamp(min=1e-8)
                    sharpness = -(w_c * w_c.log()).sum(dim=-1).mean()
                    aux_terms.append(balance + sharpness)
                aux = torch.stack(aux_terms).mean()

                loss = mse + cfg.lambda_aux * aux
                optim.zero_grad()
                loss.backward()
                optim.step()

                run.log({
                    "train/loss": loss.item(),
                    "train/mse": mse.item(),
                    "train/aux": aux.item(),
                })

            # --- validate ---
            model.eval()
            with torch.no_grad():
                pred_v = model(X_VAL_N)
                err = pred_v - Y_VAL_N
                val_p_rmse = err[..., 0].pow(2).mean().sqrt().item()
                val_u_rmse = err[..., 1].pow(2).mean().sqrt().item()
                val_v_rmse = err[..., 2].pow(2).mean().sqrt().item()
                val_vel_rmse = float(((val_u_rmse ** 2 + val_v_rmse ** 2) / 2) ** 0.5)

            run.log({
                "epoch": epoch,
                "val/pressure_rmse": val_p_rmse,
                "val/velocity_rmse": val_vel_rmse,
                "val/u_rmse": val_u_rmse,
                "val/v_rmse": val_v_rmse,
            })

        # Save this trial's fine-tuned checkpoint as a model artifact.
        tmp = Path(tempfile.mkdtemp())
        ckpt_path = tmp / "airfoil_surrogate_tuned.pt"
        torch.save(
            {
                "state_dict": model.state_dict(),
                "config": PRETRAINED_CONFIG,
                "final_val_pressure_rmse": val_p_rmse,
                "sweep_hyperparameters": dict(cfg),
            },
            ckpt_path,
        )
        art = wandb.Artifact(
            name="airfoil_surrogate_ckpt_tuned",
            type="model",
            metadata={"val_pressure_rmse": val_p_rmse, **dict(cfg)},
        )
        art.add_file(str(ckpt_path))
        run.log_artifact(art)


## 5. Launch the sweep

`wandb.sweep` registers the config with the controller and returns a
`sweep_id`. `wandb.agent` polls the controller for HP proposals and
runs `train_fn` once per proposal, up to `count` times. The controller
uses the `val/pressure_rmse` each trial logs to update its beliefs.

Open the printed URL in a browser to watch the sweep populate live.


In [5]:
sweep_id = wandb.sweep(SWEEP_CONFIG, entity=ENTITY, project=PROJECT)

print(f"sweep id  = {sweep_id}")
print(f"sweep url = https://wandb.ai/{ENTITY}/{PROJECT}/sweeps/{sweep_id}")

wandb.agent(sweep_id, function=train_fn, count=8)


Create sweep with ID: o6k2n5t8
Sweep URL: https://wandb.ai/wandb_emea/jb_physics_demo/sweeps/o6k2n5t8
sweep id  = o6k2n5t8
sweep url = https://wandb.ai/wandb_emea/jb_physics_demo/sweeps/o6k2n5t8


wandb: Agent Starting Run: yhveil4a with config:
wandb: 	batch_size: 8
wandb: 	lambda_aux: 0.008737049803566849
wandb: 	lr: 0.00045872696940823154
wandb: 	weight_decay: 4.771958291168459e-06
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch,▁▃▅▆█
train/aux,▁▅▄▃▄▄▄▃▂▅▅▅▃▄▅▄▄█▆▄▅▇▃▅▄▅▆█▅█▅█▆▆▆▄▅▅▇▅
train/loss,▃▃▅▅█▅▃▃▆▄▃▅▂▄▂▃▂▃▂▃▃▂▄▄▄▂▃▂▂▂▁▃▂▂▂▃▁▁▁▁
train/mse,▃▃▅▅█▅▃▃▆▄▃▅▂▄▂▃▂▃▂▃▃▂▄▄▄▂▃▂▂▂▁▃▂▂▂▃▁▁▁▁
val/pressure_rmse,█▅▃▂▁
val/u_rmse,█▇▄▁▁
val/v_rmse,█▇▅▃▁
val/velocity_rmse,█▇▄▂▁
epoch,4
train/aux,-0.67253
train/loss,0.30982


wandb: Agent Starting Run: 3yi8rxrl with config:
wandb: 	batch_size: 4
wandb: 	lambda_aux: 0.047789092505287255
wandb: 	lr: 0.00025289124476298667
wandb: 	weight_decay: 1.7904087717421684e-06
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch,▁▃▅▆█
train/aux,▂█▄▃▅▅▄▆▅▅▅▄▁▃▂▁▃▂▃▅▃▃▅▄▄▄▄▅▂▂▁▄▃▃▄▃▃▃▄▁
train/loss,▃█▅▄▆▇▄▄▄▅▃▃▆▄▃▁▄▃▅▄▅▅▃▂▄▅▄▃▂▄▃▃▅▂▄▃▂▃▁▃
train/mse,▅▆▇▄▆▅▇█▅▇▆▃▅█▅▃▆▆▅▄▆▄▂▄▅▆▅▄▅▃▃▁▂▅▄▃▅▂▃▄
val/pressure_rmse,█▅▃▂▁
val/u_rmse,█▇▃▁▁
val/v_rmse,█▇▄▂▁
val/velocity_rmse,█▇▃▁▁
epoch,4
train/aux,-0.87783
train/loss,0.39243


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: eauzcsya with config:
wandb: 	batch_size: 8
wandb: 	lambda_aux: 0.03719109113815896
wandb: 	lr: 0.0013437438356583384
wandb: 	weight_decay: 4.2776423055374286e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


... (7 outputs left out)

wandb: Agent Starting Run: eyr28pal with config:
wandb: 	batch_size: 16
wandb: 	lambda_aux: 0.04547580611114507
wandb: 	lr: 0.0015291417445405968
wandb: 	weight_decay: 1.0504565600170892e-06
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch,▁▃▅▆█
train/aux,▆█▇▄▄▃▃▂▁▂▄█▃▅▃▃▂▃▅▃
train/loss,▂█▄▄▄▃▃▃▄▃▂▂▂▁▂▂▁▂▁▁
train/mse,▂█▄▄▄▃▃▃▄▃▂▂▂▁▂▂▁▂▁▁
val/pressure_rmse,█▇▅▂▁
val/u_rmse,█▅▄▄▁
val/v_rmse,█▄▂▂▁
val/velocity_rmse,█▅▃▃▁
epoch,4
train/aux,-0.85483
train/loss,0.43776


wandb: Agent Starting Run: q7mo5uw7 with config:
wandb: 	batch_size: 4
wandb: 	lambda_aux: 0.018655650627646743
wandb: 	lr: 0.00014118386554413107
wandb: 	weight_decay: 1.668202064269874e-06
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


epoch,▁▃▅▆█
train/aux,▅▅▄▇▃▃▄▂█▆▆▆▄▃▂▁▅▆▃▃▆▃▅▃▇▅▂▆▇▄▄█▆▅▃▆▅▄▆█
train/loss,▅█▄▅▂▅▅▅▅▅▃▃▇▅▄▅▃▄▄▄▄▄▅▁▃▃▁▃▅▅▄▃▄▆▃▄▄▁▅▃
train/mse,▅▄▅▂▅▅▄▃▂▃▄█▆▅▃▄▆▄▃▂▄▅▃▃▅▄▂▂▃▅▅▅▂▄▆▁▂▄▅▃
val/pressure_rmse,█▇▅▃▁
val/u_rmse,█▆▄▂▁
val/v_rmse,█▆▅▄▁
val/velocity_rmse,█▆▄▃▁
epoch,4
train/aux,-0.53694
train/loss,0.29057


## 6. Promote the winner into a new Registry Collection

The best trial is the one with the lowest final `val/pressure_rmse`.
We resolve it via `wandb.Api().sweep(...)`, grab its logged model
artifact, and `run.link_artifact` it into the new Collection
`model_hyperparameter_tuned` under the existing `airfoil_surrogates`
Registry — no `create_registry` needed; Collections are created
implicitly on first link.


In [6]:
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="sweep-promotion",
    name="sweep-winner-promotion",
    tags=["notebook", "sweep", "promotion"],
) as promo_run:
    
    api = wandb.Api()
    sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sweep_id}")

    # Best trial by final val/pressure_rmse (min).
    best = min(
        sweep.runs,
        key=lambda r: r.summary.get("val/pressure_rmse", float("inf")),
    )
    
    best_rmse = best.summary.get("val/pressure_rmse")
    print(f"winner: {best.name}   val/pressure_rmse = {best_rmse:.4f}   (baseline was {baseline_p_rmse:.4f})")

    # Grab the model artifact this trial logged.
    best_art = next(a for a in best.logged_artifacts() if a.type == "model")

    target_path = f"wandb-registry-{REGISTRY_NAME}/{TUNED_COLLECTION}"
    promo_run.link_artifact(
        best_art,
        target_path=target_path,
        aliases=["production", "sweep-winner"],
    )
    
    print(f"linked {best_art.name}:{best_art.version} -> {target_path}")
    print(f"aliases: production, sweep-winner")


winner: amber-sweep-4   val/pressure_rmse = 0.4057   (baseline was 0.6750)
linked airfoil_surrogate_ckpt_tuned:v3:v3 -> wandb-registry-airfoil_surrogates/model_hyperparameter_tuned
aliases: production, sweep-winner


## 7. Wrap up — what to look at in the W&B UI

**Sweep page** (URL printed in §5)
- Parallel-coordinates plot with all four HPs and the final metric.
- Parameter-importance panel — expect `lr` to dominate.
- 8 individual trial runs, each with train/val curves.

**Lineage graph** (Artifacts tab)
- Every sweep trial run consumes both Registry entries:
  `checkpoints:production` and `datasets:production`. Setup and
  promotion runs consume the same two artifacts.

**Registry**
- Registry `airfoil_surrogates` now has two Collections:
  - `checkpoints` — base model versions from
    `experiment_tracking.ipynb`.
  - `model_hyperparameter_tuned` — the sweep winner, aliased
    `production` + `sweep-winner`.

Downstream notebooks / consumers can pull the tuned model with
`run.use_artifact("wandb-registry-airfoil_surrogates/model_hyperparameter_tuned:production")`.
